# Z-dim Selection Diagnostics

Diagnostic notebook covering D1 (per-feature NLL), D2 (posterior std at init),
and D4 (horseshoe forensics) from
[`~/.claude/plans/kind-wandering-bonbon.md`](file:///nfs/users/nfs_v/vk7/.claude/plans/kind-wandering-bonbon.md).

**Scope:**
- **Phase 5a** (this run, no Plan 1 Phase 4 models required): iterate the 6
  existing model pairs (A–F) + 2 existing horseshoe failed runs + loss-at-init
  flow on freshly-instantiated models. Validates the entire pipeline
  end-to-end before the new sweep models land.
- **Phase 5b** (after Plan 1 Phase 4 GPU jobs complete): fill in
  `PLAN1_SWEEP_MODELS` from `z_init_sigma_jobs.tsv` and re-run.

**Loading:** every model is loaded via `RegularizedMultimodalVI.load(dir, adata=mdata)`,
never from checkpoints. Per-feature NLL results are cached as parquet under
each model directory, so re-running is free.

**Figures** are saved to `figures/z_dim_selection/` as PNG (300 DPI) + PDF.

**D5 (horseshoe 4-arm comparison)** is **not** in this notebook. The final
cell prints the four Plan 1 Block 3 model directory paths so the user can
open them on wandb (see `Steps for user` in the plan).


## Papermill parameters

Parameters the notebook accepts via papermill overrides.

- `datasets_to_run` — subset to e.g. `["bm"]` to run a single dataset.
- `run_loss_at_init` — toggle the D1 loss-at-init flow (slow-ish).
- `run_d1` / `run_d2` / `run_d4` — toggle individual diagnostic phases.
- `figures_dir` / `summary_tsv` — output paths.
- `fig_suffix` — set to `""` for the Phase 5b final run so figures overwrite the Phase 5a previews.
- `tsv_path` — path to `z_init_sigma_jobs.tsv` (Plan 1 sweep registry).
- `marker_genes_csv` / `embryo_tf_list_path` — gene lists for D1 diagnostics.
- `batch_size` — inference batch size for per-feature NLL + per-dim KL.


In [ ]:
datasets_to_run = ["bm", "immune", "embryo"]
run_loss_at_init = True
run_d1 = True
run_d2 = True
run_d4 = True
figures_dir = "docs/notebooks/model_comparisons/figures/z_dim_selection"
summary_tsv = "docs/notebooks/model_comparisons/z_dim_selection_summary.tsv"
fig_suffix = "_phase5a"
tsv_path = "docs/notebooks/model_comparisons/z_init_sigma_jobs.tsv"
marker_genes_csv = "docs/notebooks/known_marker_genes.csv"
embryo_tf_list_path = (
    "/nfs/team205/vk7/sanger_projects/cell2state_embryo/data/tf_annotations/claude_searches/tf_list_281.txt"
)
batch_size = 512

### Papermill parameter coercion

`coerce_papermill_params` rewrites string values (e.g. `"0"` from CLI
overrides) into the correct Python types. Critical for booleans since
`bool("0")` is `True`.


In [ ]:
from regularizedvi.utils import coerce_papermill_params

_coerced = coerce_papermill_params(
    run_loss_at_init=(run_loss_at_init, bool),
    run_d1=(run_d1, bool),
    run_d2=(run_d2, bool),
    run_d4=(run_d4, bool),
    batch_size=(batch_size, int),
)
run_loss_at_init = _coerced["run_loss_at_init"]
run_d1 = _coerced["run_d1"]
run_d2 = _coerced["run_d2"]
run_d4 = _coerced["run_d4"]
batch_size = _coerced["batch_size"]
print(
    f"Params: run_loss_at_init={run_loss_at_init} run_d1={run_d1} "
    f"run_d2={run_d2} run_d4={run_d4} batch_size={batch_size}"
)

## Imports and paths

In [ ]:
import sys
import traceback
from pathlib import Path

import matplotlib

matplotlib.use("Agg")

import numpy as np
import pandas as pd
import torch

# Local import of the plotting helpers module (kept next to this notebook).
_helpers_dir = str(Path.cwd() / "docs/notebooks/model_comparisons")
if _helpers_dir not in sys.path:
    sys.path.insert(0, _helpers_dir)
import _z_dim_diagnostic_plots as plots  # noqa: E402

from regularizedvi import RegularizedMultimodalVI  # noqa: E402

figures_dir = Path(figures_dir)
figures_dir.mkdir(parents=True, exist_ok=True)
print(f"Figures -> {figures_dir}")
print(f"Fig suffix: {fig_suffix!r}")

## Model registries

Hardcoded paths for the **6 existing model pairs** (A–F) plus the **2 existing
horseshoe failed runs**. Plan 1 sweep models are read from
`z_init_sigma_jobs.tsv` at run time and filtered to rows whose
`model/model.pt` exists on disk — this is how Phase 5b gets picked up without
notebook edits.


In [ ]:
# -- Existing model pairs --------------------------------------------------
# Pair A klw400 is retrained by Plan 1 row 17 (`bm_mm_baseline_disp1_dwl2_01_klw400_retrain`);
# until that GPU job finishes, the Pair A klw400 slot is skipped.
EXISTING_PAIRS: dict[str, dict[str, dict[str, str | None]]] = {
    "bm": {
        "A_klw0": {
            "label": "pairA_bm_nonburst_klw0",
            "path": "/nfs/team205/vk7/sanger_projects/my_packages/regularizedvi/results/bm_mm_baseline_klw0/model",
        },
        "A_klw400": {
            "label": "pairA_bm_nonburst_klw400",
            "path": "/nfs/team205/vk7/sanger_projects/my_packages/regularizedvi/results/bm_mm_baseline_disp1_dwl2_01_klw400_retrain/model",
        },
        "B_klw0": {
            "label": "pairB_bm_burst_klw0",
            "path": "/nfs/team205/vk7/sanger_projects/my_packages/regularizedvi/results/bm_mm_burst_vbs2_int1_klw0/model",
        },
        "B_klw400": {
            "label": "pairB_bm_burst_klw400",
            "path": "/nfs/team205/vk7/sanger_projects/my_packages/regularizedvi/results/bm_mm_burst_vbs2_int1/model",
        },
    },
    "immune": {
        "C_klw0": {
            "label": "pairC_immune_nonburst_klw0",
            "path": "/nfs/team205/vk7/sanger_projects/my_packages/regularizedvi/results/immune_baseline_large_klw0_ld_bs2k/model",
        },
        "C_klw400": {
            "label": "pairC_immune_nonburst_klw400",
            "path": "/nfs/team205/vk7/sanger_projects/my_packages/regularizedvi/results/immune_integration_rna_v2/model",
        },
        "D_klw0": {
            "label": "pairD_immune_burst_klw0",
            "path": "/nfs/team205/vk7/sanger_projects/my_packages/regularizedvi/results/immune_burst_vbs2_int1_large_klw0_ld_bs2k/model",
        },
        "D_klw400": {
            "label": "pairD_immune_burst_klw400",
            "path": "/nfs/team205/vk7/sanger_projects/my_packages/regularizedvi/results/immune_burst_vbs2_int1_large/model",
        },
    },
    "embryo": {
        "E_klw0": {
            "label": "pairE_embryo_nonburst_klw0",
            "path": "/nfs/team283/vk7/sanger_projects/cell2state_embryo/results/regularizedvi_embryo_v1/embryo_baseline_klw0/model",
        },
        "E_klw400": {
            "label": "pairE_embryo_nonburst_klw400",
            "path": "/nfs/team283/vk7/sanger_projects/cell2state_embryo/results/regularizedvi_embryo_v1/embryo_disp_data_prior1_gpunormal/model",
        },
        "F_klw0": {
            "label": "pairF_embryo_burst_klw0",
            "path": "/nfs/team283/vk7/sanger_projects/cell2state_embryo/results/regularizedvi_embryo_v1/embryo_burst_vbs2_int1_n_klw0/model",
        },
        "F_klw400": {
            "label": "pairF_embryo_burst_klw400",
            "path": "/nfs/team283/vk7/sanger_projects/cell2state_embryo/results/regularizedvi_embryo_v1/embryo_burst_vbs2_int1_n/model",
        },
    },
}

# -- Existing horseshoe LN-Gamma failed runs (for D4 forensics) -----------
HORSESHOE_FAILED = {
    "bm": {
        "label": "horseshoe_ln_gamma_bm_failed",
        "path": "/nfs/team205/vk7/sanger_projects/my_packages/regularizedvi/results/bm_mm_horseshoe_ln_baseline_klw0/model",
    },
    "immune": {
        "label": "horseshoe_ln_gamma_immune_failed",
        "path": "/nfs/team205/vk7/sanger_projects/my_packages/regularizedvi/results/immune_horseshoe_ln_baseline_large_klw0/model",
    },
}

In [ ]:
# Read Plan 1 sweep registry from z_init_sigma_jobs.tsv. Each row's
# results_folder/model/ is the load path; rows with no model.pt yet are skipped.
sweep_tsv = Path(tsv_path)
if not sweep_tsv.exists():
    print(f"[warn] sweep TSV not found: {sweep_tsv}")
    sweep_df = pd.DataFrame()
else:
    sweep_df = pd.read_csv(sweep_tsv, sep="\t")
    print(f"Loaded sweep TSV with {len(sweep_df)} rows")


def sweep_row_to_entry(row: pd.Series) -> dict[str, str] | None:
    """Extract (label, path, exists) from a sweep TSV row."""
    name = row.get("name")
    rf = row.get("results_folder")
    if not isinstance(rf, str) or not isinstance(name, str):
        return None
    # Absolute-vs-relative: the BM/immune rows are relative to the package root;
    # the embryo row uses an absolute path (results/embryo...)
    path = Path(rf)
    if not path.is_absolute():
        path = Path.cwd() / path
    mp = path / "model"
    pt = mp / "model.pt"
    return {"label": name, "path": str(mp), "exists": pt.exists()}


def sweep_models_for_dataset(dataset: str) -> dict[str, dict[str, str]]:
    """Return sweep-row model dirs for a dataset whose model.pt exists on disk."""
    if sweep_df.empty:
        return {}
    if dataset == "bm":
        mask = sweep_df["name"].astype(str).str.startswith("bm_")
    elif dataset == "immune":
        mask = sweep_df["name"].astype(str).str.startswith("immune_")
    elif dataset == "embryo":
        mask = sweep_df["name"].astype(str).str.startswith("embryo_")
    else:
        return {}
    out: dict[str, dict[str, str]] = {}
    for _, row in sweep_df[mask].iterrows():
        entry = sweep_row_to_entry(row)
        if entry is None:
            continue
        label = entry["label"]
        if not entry["exists"]:
            print(f"  [skip] {label} — model.pt not found at {entry['path']}")
            continue
        out[label] = {"label": label, "path": entry["path"]}
        print(f"  [ok]   {label}")
    return out


print("Sweep model coverage per dataset:")
for ds in datasets_to_run:
    print(f"--- {ds} ---")
    sweep_models_for_dataset(ds)

## Marker gene + TF gene lists

In [ ]:
marker_genes: list[str] = []
tf_genes: list[str] = []

if Path(marker_genes_csv).exists():
    _mg_df = pd.read_csv(marker_genes_csv)
    if "gene" in _mg_df.columns:
        marker_genes = sorted(set(_mg_df["gene"].dropna().astype(str).tolist()))
    print(f"Loaded {len(marker_genes)} marker genes from {marker_genes_csv}")
else:
    print(f"[warn] marker genes CSV not found: {marker_genes_csv}")

if Path(embryo_tf_list_path).exists():
    with open(embryo_tf_list_path) as f:
        tf_genes = [line.strip() for line in f if line.strip()]
    print(f"Loaded {len(tf_genes)} TF genes from {embryo_tf_list_path}")
else:
    print(f"[warn] TF list not found: {embryo_tf_list_path}")

GENE_SETS = {
    "all": None,
    "markers": marker_genes if marker_genes else None,
    "tfs": tf_genes if tf_genes else None,
}
# Drop entries that are still None-valued (missing file) but preserve "all".
GENE_SETS = {k: v for k, v in GENE_SETS.items() if k == "all" or v is not None}
print(f"Gene sets available: {list(GENE_SETS.keys())}")

## Per-dataset loaders

Each loader returns a MuData registered via the same `setup_mudata` call
used by the training notebook for that dataset, so `Model.load(..., adata=mdata)`
works cleanly.


In [ ]:
def load_bm_mdata():
    """Load BM multiome and rebuild the MuData used by the training notebook.

    Mirrors the preprocessing in
    `docs/notebooks/model_comparisons/bone_marrow_multimodal_tutorial_early_stopping.ipynb`.
    """
    import mudata as mu
    import scanpy as sc

    from regularizedvi.utils import download_bone_marrow_dataset, filter_genes

    h5ad_path = download_bone_marrow_dataset(data_folder="data/")
    adata_full = sc.read_h5ad(h5ad_path)

    gex_mask = adata_full.var["feature_types"] == "GEX"
    adata_full.var["SYMBOL"] = adata_full.var_names.values.copy()
    gex_adata = adata_full[:, gex_mask].copy()
    gex_adata.var["mt"] = [g.startswith("MT-") for g in gex_adata.var["SYMBOL"]]
    mt_counts = gex_adata[:, gex_adata.var["mt"].tolist()].X.sum(1)
    mt_counts = np.asarray(mt_counts).squeeze()
    adata_full.obs["mt_frac"] = mt_counts / adata_full.obs["GEX_n_counts"]
    del gex_adata

    cell_mask = (
        (adata_full.obs["GEX_n_genes"] > 500)
        & (adata_full.obs["GEX_n_counts"] > 1000)
        & (adata_full.obs["GEX_n_counts"] < 80000)
        & (adata_full.obs["GEX_n_genes"] < 10000)
        & (adata_full.obs["ATAC_atac_fragments"] > 2000)
        & (adata_full.obs["ATAC_atac_fragments"] < 100000)
        & (adata_full.obs["mt_frac"] < 0.20)
    )
    adata_full = adata_full[cell_mask, :].copy()

    gex_mask = adata_full.var["feature_types"] == "GEX"
    atac_mask = adata_full.var["feature_types"] == "ATAC"
    adata_rna = adata_full[:, gex_mask].copy()
    adata_atac = adata_full[:, atac_mask].copy()
    adata_rna.X = adata_rna.layers["counts"]
    adata_atac.X = adata_atac.layers["counts"]
    adata_rna.var["SYMBOL"] = adata_rna.var_names.values.copy()
    adata_rna.var_names = adata_rna.var["gene_ids"].values.astype(str).copy()

    sel_rna = filter_genes(adata_rna, cell_count_cutoff=15, cell_percentage_cutoff2=0.01, nonz_mean_cutoff=1.03)
    adata_rna = adata_rna[:, sel_rna].copy()
    sel_atac = filter_genes(adata_atac, cell_count_cutoff=15, cell_percentage_cutoff2=0.01, nonz_mean_cutoff=1.03)
    adata_atac = adata_atac[:, sel_atac].copy()

    mdata = mu.MuData({"rna": adata_rna, "atac": adata_atac})
    RegularizedMultimodalVI.setup_mudata(
        mdata,
        ambient_covariate_keys=["batch"],
        nn_conditioning_covariate_keys=["site", "donor"],
        feature_scaling_covariate_keys=["site", "donor"],
        dispersion_key="batch",
        library_size_key="batch",
        encoder_covariate_keys=False,
    )
    print(f"BM mdata: {mdata}")
    return mdata


def load_immune_mdata():
    """Load immune RNA-only and build MuData with a single 'rna' modality.

    Mirrors the preprocessing in
    `docs/notebooks/immune_integration/bm_pbmc_rna_training_v3.ipynb`.
    """
    import mudata as mu
    import scanpy as sc

    rna_path = "/nfs/team205/vk7/sanger_projects/my_packages/regularizedvi/results/immune_integration/adata_rna.h5ad"
    adata = sc.read_h5ad(rna_path)
    print(f"Immune adata: {adata.shape}")

    adata_for_mdata = adata.copy()
    adata_for_mdata.X = adata_for_mdata.layers["counts"]
    mdata = mu.MuData({"rna": adata_for_mdata})

    use_feature_scaling = True
    RegularizedMultimodalVI.setup_mudata(
        mdata,
        ambient_covariate_keys=["batch"],
        nn_conditioning_covariate_keys=["dataset", "donor"],
        feature_scaling_covariate_keys=["dataset", "donor"] if use_feature_scaling else [],
        dispersion_key="batch",
        library_size_key="batch",
        encoder_covariate_keys=False,
    )
    return mdata


def load_embryo_mdata():
    """Load embryo 4-modality (RNA + spliced + unspliced + ATAC) MuData.

    Mirrors the preprocessing in
    `/nfs/team205/vk7/sanger_projects/cell2state_embryo/notebooks/benchmark/
    regularizedvi/embryo_rna_atac_spliced_unspliced.ipynb` — without the
    optional compound QC filter or per-modality gene re-filtering, since
    the trained models we are loading were already registered with fully
    filtered var_names. For loading via Model.load(..., adata=mdata) the
    obs/var axes must match the trained model, so we skip extra filtering
    and rely on the saved model's adata_manager.
    """
    import gc
    import re

    import anndata as ad
    import mudata as mu
    import scanpy as sc

    results_folder_scvi = (
        "/nfs/team283/vk7/sanger_projects/cell2state_embryo/results/"
        "scvi_customV1_sn_reseq2_njc_genes28k_batch1024_nn3000latent700layer1_1000epochs_"
        "NucleiTech_embryo_Nhid16Var02_add_NoBatchDecoder_dropIN_DispMAP"
    )
    rna_h5ad_path = f"{results_folder_scvi}/outputs/transferred_rna_v6.h5ad"
    atac_h5ad_path = f"{results_folder_scvi}/outputs/topn539781_bin1000_atac_05_05_2024.h5ad"
    velocyto_path = "/nfs/team283/vk7/sanger_projects/large_data/embryo_multiome/velocyto_spliced_unspliced.h5ad"

    print(f"Reading {rna_h5ad_path} ...")
    adata_rna = sc.read_h5ad(rna_h5ad_path)

    # Normalise obs_names: {barcode}_{sample_id} -> {sample_id}-{barcode}
    _bc = re.compile(r"^([ACGT]{16}-1)_(.+)$")
    new_names = []
    for name in adata_rna.obs_names:
        m = _bc.match(name)
        if m:
            new_names.append(f"{m.group(2)}-{m.group(1)}")
        else:
            new_names.append(name)
    adata_rna.obs_names = pd.Index(new_names)

    print(f"Reading {velocyto_path} ...")
    adata_vel = sc.read_h5ad(velocyto_path)
    common_vel = adata_rna.obs_names.intersection(adata_vel.obs_names)
    adata_vel = adata_vel[common_vel].copy()
    adata_spliced = ad.AnnData(
        X=adata_vel.layers["spliced"].copy(),
        obs=adata_vel.obs.copy(),
        var=adata_vel.var.copy(),
    )
    adata_spliced.obs_names = adata_vel.obs_names.copy()
    adata_spliced.var_names = adata_vel.var_names.copy()
    adata_unspliced = ad.AnnData(
        X=adata_vel.layers["unspliced"].copy(),
        obs=adata_vel.obs.copy(),
        var=adata_vel.var.copy(),
    )
    adata_unspliced.obs_names = adata_vel.obs_names.copy()
    adata_unspliced.var_names = adata_vel.var_names.copy()
    del adata_vel
    gc.collect()

    print(f"Reading {atac_h5ad_path} (backed) ...")
    adata_atac = sc.read_h5ad(atac_h5ad_path, backed="r")
    adata_atac = adata_atac.to_memory()
    if "counts_120" in adata_atac.layers:
        adata_atac.X = adata_atac.layers["counts_120"].copy()

    # Align on the intersection of all four modalities' obs_names.
    common_cells = adata_rna.obs_names.intersection(adata_atac.obs_names).intersection(adata_spliced.obs_names)
    print(f"Common cells across 4 modalities: {len(common_cells)}")
    adata_rna = adata_rna[common_cells, :].copy()
    adata_spliced = adata_spliced[common_cells, :].copy()
    adata_unspliced = adata_unspliced[common_cells, :].copy()
    adata_atac = adata_atac[common_cells, :].copy()
    adata_rna.X = adata_rna.layers["counts"]

    # Copy rna obs metadata to spliced/unspliced so setup_mudata sees them.
    for col in adata_rna.obs.columns:
        if col not in adata_spliced.obs.columns:
            adata_spliced.obs[col] = adata_rna.obs[col].values
        if col not in adata_unspliced.obs.columns:
            adata_unspliced.obs[col] = adata_rna.obs[col].values

    mdata = mu.MuData(
        {
            "rna": adata_rna,
            "spliced": adata_spliced,
            "unspliced": adata_unspliced,
            "atac": adata_atac,
        }
    )
    use_feature_scaling = True
    RegularizedMultimodalVI.setup_mudata(
        mdata,
        ambient_covariate_keys=["sample_id"],
        nn_conditioning_covariate_keys=["Embryo", "10x_kit"],
        feature_scaling_covariate_keys=["Embryo", "10x_kit"] if use_feature_scaling else [],
        dispersion_key="sample_id",
        library_size_key="sample_id",
        encoder_covariate_keys=False,
    )
    print(f"Embryo mdata: {mdata}")
    return mdata


DATASET_LOADERS = {
    "bm": load_bm_mdata,
    "immune": load_immune_mdata,
    "embryo": load_embryo_mdata,
}

## Model iteration helpers

In [ ]:
def _collect_models_for_dataset(dataset: str) -> dict[str, dict[str, str]]:
    """Combine existing pairs + horseshoe failed run + sweep rows for a dataset."""
    out: dict[str, dict[str, str]] = {}
    for _key, meta in EXISTING_PAIRS.get(dataset, {}).items():
        pt = Path(meta["path"]) / "model.pt"
        if pt.exists():
            out[meta["label"]] = meta
        else:
            print(f"  [skip] {meta['label']} — model.pt not found at {pt}")
    hs = HORSESHOE_FAILED.get(dataset)
    if hs is not None and (Path(hs["path"]) / "model.pt").exists():
        out[hs["label"]] = hs
    out.update(sweep_models_for_dataset(dataset))
    return out


def _load_model(path: str, mdata):
    """Load a model, returning None if anything goes wrong."""
    try:
        return RegularizedMultimodalVI.load(path, adata=mdata)
    except Exception as exc:  # noqa: BLE001
        print(f"[warn] failed to load {path}: {exc}")
        return None


def _compute_per_feature_nll_safe(model, save_dir: str):
    """Per-feature NLL with graceful degradation."""
    try:
        return model.get_per_feature_reconstruction_loss(batch_size=batch_size, save_dir=save_dir)
    except Exception:  # noqa: BLE001
        traceback.print_exc()
        return None


def _compute_per_dim_kl_safe(model):
    """Per-dim KL with graceful degradation."""
    try:
        return model.get_per_dim_kl(batch_size=batch_size)
    except Exception:  # noqa: BLE001
        traceback.print_exc()
        return None


def _compute_per_feature_nll(model, save_dir: str) -> dict[str, pd.Series]:
    """Wrapper around `get_per_feature_reconstruction_loss` that also caches parquet."""
    return model.get_per_feature_reconstruction_loss(batch_size=batch_size, save_dir=save_dir)


def _compute_per_dim_kl(model) -> dict[str, np.ndarray]:
    return model.get_per_dim_kl(batch_size=batch_size)


def _extract_init_params(model) -> dict:
    """Extract constructor kwargs from a trained model for loss-at-init flow."""
    init = dict(getattr(model, "init_params_", {}).get("non_kwargs", {}))
    init.update(dict(getattr(model, "init_params_", {}).get("kwargs", {})))
    # Remove anything that can't be serialised as a constructor arg.
    init.pop("adata", None)
    init.pop("mdata", None)
    init.pop("adata_manager", None)
    return init


def _fresh_model_from_params(mdata, init_params: dict, overrides: dict | None = None):
    kwargs = dict(init_params)
    if overrides is not None:
        kwargs.update(overrides)
    return RegularizedMultimodalVI(mdata, **kwargs)


def _variant_vis_key(model) -> object:
    """Return the var_init_scale value for a trained model (used to group variants for loss-at-init)."""
    params = getattr(model, "init_params_", {})
    merged = dict(params.get("non_kwargs", {}))
    merged.update(dict(params.get("kwargs", {})))
    return merged.get("var_init_scale", "default")

## D1 / D2 / D4 compute and plot (per dataset)

The notebook processes one dataset at a time: load mdata, iterate models,
compute per-feature NLL + per-dim KL, run D1/D2/D4 plots. Results are
accumulated in `PER_DATASET_RESULTS` for the cross-dataset section.


In [ ]:
# Per-dataset aggregated result containers.
PER_DATASET_RESULTS: dict[str, dict] = {}


def process_dataset(dataset: str) -> None:
    """Load the dataset, iterate its models, compute D1/D2/D4 per-model artefacts, plot."""
    print(f"\n=== {dataset} ===")

    try:
        mdata = DATASET_LOADERS[dataset]()
    except NotImplementedError as exc:
        print(f"[skip] {dataset}: {exc}")
        return
    except Exception:  # noqa: BLE001
        print(f"[error] {dataset} loading failed:")
        traceback.print_exc()
        return

    models = _collect_models_for_dataset(dataset)
    if not models:
        print(f"[skip] {dataset}: no models found")
        return

    nll_per_variant: dict[str, dict[str, pd.Series]] = {}
    kl_per_variant: dict[str, dict[str, np.ndarray]] = {}
    active_dim_per_variant: dict[str, dict[str, float]] = {}

    loaded_models: dict[str, object] = {}
    init_params_reference: dict | None = None

    for label, meta in models.items():
        print(f"--- loading {label}")
        model = _load_model(meta["path"], mdata)
        if model is None:
            continue
        loaded_models[label] = model
        if init_params_reference is None:
            init_params_reference = _extract_init_params(model)

        if run_d1:
            # Gate NLL and KL together: if either fails, skip the pair so
            # the dicts stay in sync for downstream plotting.
            try:
                nll_dict = _compute_per_feature_nll(model, meta["path"])
                kl_dict = _compute_per_dim_kl(model)
            except Exception:  # noqa: BLE001
                print(f"[warn] D1 compute failed for {label}")
                traceback.print_exc()
            else:
                nll_per_variant[label] = nll_dict
                kl_per_variant[label] = kl_dict

        # Active dim counts from model.history_: only the canonical
        # n_active_dims_{name}_train keys (filtered by modality_names to
        # drop kl_only_ / zgamma_only_ breakdown variants).
        if getattr(model, "history_", None):
            active = {}
            mod_whitelist = set(getattr(model.module, "modality_names", []))
            for key, df in model.history_.items():
                if not (key.startswith("n_active_dims_") and key.endswith("_train")):
                    continue
                modality = key[len("n_active_dims_") : -len("_train")]
                if modality not in mod_whitelist:
                    continue
                try:
                    active[modality] = float(df.iloc[-1].values[0])
                except Exception:  # noqa: BLE001
                    pass
            if active:
                active_dim_per_variant[label] = active

    PER_DATASET_RESULTS[dataset] = {
        "mdata": mdata,
        "models": loaded_models,
        "nll_per_variant": nll_per_variant,
        "kl_per_variant": kl_per_variant,
        "active_dim_per_variant": active_dim_per_variant,
        "init_params_reference": init_params_reference,
    }

    # ---- D1 plots ------------------------------------------------------
    if run_d1 and nll_per_variant:
        try:
            plots.plot_per_feature_nll_panel(
                nll_per_variant,
                kl_per_dim_per_variant=kl_per_variant,
                gene_sets=GENE_SETS,
                output_dir=figures_dir,
                stem=f"d1_per_feature_nll_panel_{dataset}",
                suffix=fig_suffix,
            )
        except Exception:  # noqa: BLE001
            traceback.print_exc()

        if kl_per_variant:
            try:
                _, ratio_df = plots.plot_per_gene_ratio_histogram(
                    nll_per_variant,
                    kl_per_variant,
                    gene_set=(marker_genes + tf_genes) or None,
                    gene_set_name="markers+tfs",
                    output_dir=figures_dir,
                    stem=f"d1_per_gene_ratio_{dataset}",
                    suffix=fig_suffix,
                )
                PER_DATASET_RESULTS[dataset]["ratio_summary"] = ratio_df
            except Exception:  # noqa: BLE001
                traceback.print_exc()

    # ---- D1 loss-at-init + post-training improvement --------------------
    if run_d1 and run_loss_at_init and loaded_models and nll_per_variant:
        _run_loss_at_init_for_dataset(dataset, mdata, loaded_models, nll_per_variant, kl_per_variant)

    # ---- D2 plots ------------------------------------------------------
    if run_d2 and init_params_reference is not None:
        _run_d2_for_dataset(dataset, mdata, init_params_reference, active_dim_per_variant)

    # ---- D4 plots ------------------------------------------------------
    if run_d4:
        _run_d4_for_dataset(dataset, mdata, loaded_models, init_params_reference)

In [ ]:
def _run_loss_at_init_for_dataset(
    dataset,
    mdata,
    loaded_models,
    nll_per_variant,
    kl_per_variant,
):
    """Compute ΔNLL = NLL_init − NLL_trained per variant and plot D1 post-training improvement.

    Variants are grouped by their `var_init_scale` init_params_ value. One
    fresh model is instantiated per group (not per variant), so the cost is
    O(n_groups) rather than O(n_variants). Each fresh model is run in eval
    mode through the full mdata via get_per_feature_reconstruction_loss,
    which writes a parquet cache under a scratch subdir.
    """
    print(f"\n--- D1 loss-at-init ({dataset}) ---")

    # Group variants by var_init_scale.
    groups: dict[object, list[str]] = {}
    init_params_per_variant: dict[str, dict] = {}
    for label, model in loaded_models.items():
        if label not in nll_per_variant:
            continue
        vis = _variant_vis_key(model)
        groups.setdefault(vis, []).append(label)
        init_params_per_variant[label] = _extract_init_params(model)

    if not groups:
        print("[skip] no variants eligible for loss-at-init")
        return

    # Compute loss-at-init NLL per group. Reuse the init_params from the
    # first variant in each group to instantiate a matching fresh model.
    nll_init_per_group: dict[object, dict[str, pd.Series]] = {}
    for vis, labels_in_group in groups.items():
        ref_label = labels_in_group[0]
        ref_params = init_params_per_variant[ref_label]
        try:
            torch.manual_seed(42)
            np.random.seed(42)
            fresh = _fresh_model_from_params(mdata, ref_params)
            # Use a scratch save_dir per group so parquet caches don't clobber
            # the trained-model caches.
            scratch = Path(figures_dir).parent / f"loss_at_init_cache/{dataset}_vis{vis}"
            scratch.mkdir(parents=True, exist_ok=True)
            fresh.module.eval()
            nll_init_per_group[vis] = fresh.get_per_feature_reconstruction_loss(
                batch_size=batch_size, save_dir=str(scratch)
            )
            print(f"  vis={vis}: loss-at-init cached at {scratch}")
        except Exception:  # noqa: BLE001
            print(f"[warn] loss-at-init failed for vis={vis}")
            traceback.print_exc()

    # Build nll_init_per_variant by copying the group's reference to each
    # member variant (all members share the same var_init_scale).
    nll_init_per_variant: dict[str, dict[str, pd.Series]] = {}
    for vis, labels_in_group in groups.items():
        init_dict = nll_init_per_group.get(vis)
        if init_dict is None:
            continue
        for label in labels_in_group:
            nll_init_per_variant[label] = init_dict

    if not nll_init_per_variant:
        print("[skip] no loss-at-init references computed")
        return

    # Mean per-dim KL per variant × modality (precomputed in main loop).
    mean_kl_per_dim: dict[str, dict[str, float]] = {}
    for variant, kl_dict in kl_per_variant.items():
        mean_kl_per_dim[variant] = {mod: float(np.mean(arr)) for mod, arr in kl_dict.items()}

    # Filter nll_per_variant to variants that have a matching init reference.
    nll_trained_filtered = {v: nll_per_variant[v] for v in nll_init_per_variant if v in nll_per_variant}

    try:
        _, improv_df = plots.plot_post_training_improvement(
            nll_init_per_variant,
            nll_trained_filtered,
            mean_kl_per_dim,
            output_dir=figures_dir,
            stem=f"d1_post_training_improvement_{dataset}",
            suffix=fig_suffix,
        )
        PER_DATASET_RESULTS[dataset]["improvement_summary"] = improv_df
        print(improv_df)
    except Exception:  # noqa: BLE001
        traceback.print_exc()

In [ ]:
def _qz_from_fresh_model(mdata, init_params: dict, var_init_scale):
    """Instantiate a fresh model and collect qz.loc / qz.scale across the full mdata.

    Returns (qz_loc, qz_scale) as numpy arrays of shape (n_cells, total_n_latent).
    """
    overrides = {
        "var_init_scale": var_init_scale,
        "use_softplus_var_activation": True,
    }
    torch.manual_seed(42)
    np.random.seed(42)
    model = _fresh_model_from_params(mdata, init_params, overrides=overrides)
    module = model.module
    module.eval()
    device = next(module.parameters()).device

    loc_parts: list[np.ndarray] = []
    scale_parts: list[np.ndarray] = []
    scdl = model._make_data_loader(adata=mdata, batch_size=batch_size)
    with torch.inference_mode():
        for tensors in scdl:
            tensors = {k: (v.to(device) if isinstance(v, torch.Tensor) else v) for k, v in tensors.items()}
            inf_inputs = module._get_inference_input(tensors)
            out = module.inference(**inf_inputs)
            # Multimodal inference output: {"qz_per_modality": {modality: Normal}, ...}
            qz_pm = out.get("qz_per_modality", {})
            per_batch_loc: list[np.ndarray] = []
            per_batch_scale: list[np.ndarray] = []
            for name in sorted(module.modality_names):
                qz = qz_pm.get(name)
                if qz is None:
                    continue
                per_batch_loc.append(qz.loc.detach().cpu().numpy())
                per_batch_scale.append(qz.scale.detach().cpu().numpy())
            if not per_batch_loc:
                continue
            loc_parts.append(np.concatenate(per_batch_loc, axis=-1))
            scale_parts.append(np.concatenate(per_batch_scale, axis=-1))

    if not loc_parts:
        raise RuntimeError("No qz collected — module.inference output shape not recognised")
    return np.concatenate(loc_parts, axis=0), np.concatenate(scale_parts, axis=0)


def _run_d2_for_dataset(dataset, mdata, init_params, active_dim_per_variant):
    """D2: analytic init demo + variability ratio + active dim counts."""
    print(f"\n--- D2 ({dataset}) ---")
    qz_scales_per_vis: dict[object, np.ndarray] = {}
    variability_summaries: dict[object, dict[str, float]] = {}

    for vis in [None, 0.1]:
        try:
            loc, scale = _qz_from_fresh_model(mdata, init_params, vis)
        except Exception:  # noqa: BLE001
            print(f"[warn] D2 fresh-model run failed for vis={vis}")
            traceback.print_exc()
            continue
        qz_scales_per_vis[vis] = scale.ravel()
        try:
            _, summary = plots.plot_variability_ratio(
                loc,
                scale,
                dataset_label=dataset,
                var_init_scale=vis,
                output_dir=figures_dir,
                suffix=fig_suffix,
            )
            variability_summaries[vis] = summary
            print(f"  vis={vis}: {summary}")
        except Exception:  # noqa: BLE001
            traceback.print_exc()

    if qz_scales_per_vis:
        try:
            plots.plot_qz_scale_init(
                qz_scales_per_vis,
                dataset_label=dataset,
                output_dir=figures_dir,
                suffix=fig_suffix,
            )
        except Exception:  # noqa: BLE001
            traceback.print_exc()

    PER_DATASET_RESULTS[dataset]["variability_summaries"] = variability_summaries

    if active_dim_per_variant:
        try:
            plots.plot_active_dim_counts(
                active_dim_per_variant,
                output_dir=figures_dir,
                stem=f"d2_active_dim_counts_{dataset}",
                suffix=fig_suffix,
            )
        except Exception:  # noqa: BLE001
            traceback.print_exc()

In [ ]:
def _qlam_from_horseshoe_model(mdata, init_params: dict, overrides: dict):
    """Return (qlam_loc, qlam_scale) arrays for a fresh horseshoe model at init."""
    torch.manual_seed(42)
    np.random.seed(42)
    model = _fresh_model_from_params(mdata, init_params, overrides=overrides)
    module = model.module
    if not hasattr(module, "horseshoe_encoders"):
        return None, None
    module.eval()
    # Extract from one minibatch's inference (horseshoe q(lam) is encoder-driven).
    device = next(module.parameters()).device
    scdl = model._make_data_loader(adata=mdata, batch_size=min(batch_size, 4096))
    locs: list[np.ndarray] = []
    scales: list[np.ndarray] = []
    with torch.inference_mode():
        for tensors in scdl:
            tensors = {k: (v.to(device) if isinstance(v, torch.Tensor) else v) for k, v in tensors.items()}
            inf_inputs = module._get_inference_input(tensors)
            out = module.inference(**inf_inputs)
            hs_pm = out.get("horseshoe_per_modality", {})
            for name in sorted(module.modality_names):
                qlam = hs_pm.get(name)
                if qlam is None:
                    continue
                locs.append(qlam.loc.detach().cpu().numpy())
                scales.append(qlam.scale.detach().cpu().numpy())
            break  # one batch is enough for init characterisation
    if not locs:
        return None, None
    return np.concatenate(locs, axis=0), np.concatenate(scales, axis=0)


def _run_d4_for_dataset(dataset, mdata, loaded_models, init_params):
    """D4: horseshoe forensics on existing failed run + paired init characterisation."""
    print(f"\n--- D4 ({dataset}) ---")
    hs_failed_label = HORSESHOE_FAILED.get(dataset, {}).get("label")
    hs_model = loaded_models.get(hs_failed_label) if hs_failed_label else None

    # Paired init characterisation across normal + horseshoe-old + horseshoe-fixed.
    if init_params is not None:
        init_data: dict[str, dict[str, np.ndarray]] = {}
        for vis in [None, 0.1]:
            try:
                loc, scale = _qz_from_fresh_model(mdata, init_params, vis)
                key = f"normal_vis_{vis}"
                init_data[key] = {"qz_loc": loc, "qz_scale": scale}
            except Exception:  # noqa: BLE001
                print(f"[warn] D4 normal fresh-model failed for vis={vis}")
                traceback.print_exc()

        for label, overrides in [
            (
                "horseshoe_old",
                {
                    "horseshoe_latent_z_prior_type": "lognormal",
                    "var_init_scale": None,
                    "use_softplus_var_activation": False,
                },
            ),
            (
                "horseshoe_fixed",
                {
                    "horseshoe_latent_z_prior_type": "lognormal",
                    "var_init_scale": 0.1,
                    "use_softplus_var_activation": True,
                    "horseshoe_posterior_init_loc": 1.0,
                    "horseshoe_posterior_init_scale": 0.1,
                },
            ),
        ]:
            try:
                loc, scale = _qlam_from_horseshoe_model(mdata, init_params, overrides)
                if loc is not None:
                    init_data[label] = {"qlam_loc": loc, "qlam_scale": scale}
            except Exception:  # noqa: BLE001
                print(f"[warn] D4 horseshoe fresh-model failed for {label}")
                traceback.print_exc()

        if init_data:
            try:
                plots.plot_horseshoe_init_paired(
                    init_data,
                    dataset_label=dataset,
                    output_dir=figures_dir,
                    suffix=fig_suffix,
                )
            except Exception:  # noqa: BLE001
                traceback.print_exc()

    # z_gamma per-dim summary for the existing horseshoe LN-Gamma run: run the
    # trained model over mdata and average the sampled z_gamma per dim.
    if hs_model is not None:
        try:
            zgamma_accum: dict[str, list[np.ndarray]] = {}
            module = hs_model.module
            module.eval()
            scdl = hs_model._make_data_loader(adata=mdata, batch_size=batch_size)
            with torch.inference_mode():
                for tensors in scdl:
                    device = next(module.parameters()).device
                    tensors = {k: (v.to(device) if isinstance(v, torch.Tensor) else v) for k, v in tensors.items()}
                    inf_inputs = module._get_inference_input(tensors)
                    out = module.inference(**inf_inputs)
                    z_gamma_pm = out.get("z_gamma_per_modality", {})
                    for name, arr in z_gamma_pm.items():
                        zgamma_accum.setdefault(name, []).append(arr.detach().cpu().numpy())
            # Prefix with model label so the legend reads `{model}/{modality}`.
            means = {
                f"{hs_failed_label}/{modality}": np.concatenate(parts, axis=0).mean(axis=0)
                for modality, parts in zgamma_accum.items()
            }
            if means:
                plots.plot_horseshoe_zgamma_per_dim(
                    means,
                    dataset_label=dataset,
                    output_dir=figures_dir,
                    suffix=fig_suffix,
                )
        except Exception:  # noqa: BLE001
            traceback.print_exc()

        # Horseshoe KL contribution trace from model.history_ — does it
        # dominate reconstruction? (Returns None if metrics not logged.)
        try:
            plots.plot_horseshoe_kl_trace(
                hs_model.history_,
                dataset_label=dataset,
                output_dir=figures_dir,
                suffix=fig_suffix,
            )
        except Exception:  # noqa: BLE001
            traceback.print_exc()

## Run the per-dataset loop

In [ ]:
for _ds in datasets_to_run:
    try:
        process_dataset(_ds)
    except Exception:  # noqa: BLE001
        print(f"[error] {_ds} failed:")
        traceback.print_exc()

## D2 verdict — variability ratio 1000× claim

Per-dataset × per-`var_init_scale` median ratio of
`std_over_cells(qz.loc) / mean_over_cells(qz.scale)`. `claim_supported_1000x`
tests the plan's strict 1000× claim (median < 1e-3); `claim_supported_100x`
is the looser 100× form.


In [ ]:
from IPython.display import Markdown, display

_d2_rows: list[str] = []
_d2_rows.append("| dataset | var_init_scale | median_ratio | claim_100x | claim_1000x | n_dims |")
_d2_rows.append("|---|---|---|---|---|---|")
for ds, results in PER_DATASET_RESULTS.items():
    summaries = results.get("variability_summaries", {})
    for vis, s in summaries.items():
        _d2_rows.append(
            f"| {ds} | {vis} | {s['median_ratio']:.3e} | "
            f"{s.get('claim_supported_100x', '—')} | "
            f"{s.get('claim_supported_1000x', '—')} | {s['n_dims']} |"
        )
if len(_d2_rows) > 2:
    display(Markdown("\n".join(_d2_rows)))
else:
    print("[skip] no D2 variability summaries collected — no datasets ran D2")

## Cross-dataset D1 overlay

In [ ]:
# Cross-dataset comparison on the shared marker+TF set (hypothesis 2: dilution).
shared_gene_set = (marker_genes + tf_genes) or None
cross_nll: dict[str, dict[str, pd.Series]] = {}
for ds, results in PER_DATASET_RESULTS.items():
    variant_pick = None
    for candidate in results.get("nll_per_variant", {}):
        if "klw0" in candidate:
            variant_pick = candidate
            break
    if variant_pick is None and results.get("nll_per_variant"):
        variant_pick = next(iter(results["nll_per_variant"]))
    if variant_pick is not None:
        cross_nll[ds] = results["nll_per_variant"][variant_pick]

if cross_nll:
    try:
        plots.plot_cross_dataset_nll(
            cross_nll,
            modality_name="rna",
            gene_set=shared_gene_set,
            gene_set_name="markers+tfs",
            output_dir=figures_dir,
            suffix=fig_suffix,
        )
    except Exception:  # noqa: BLE001
        traceback.print_exc()
else:
    print("[skip] cross-dataset D1: no datasets produced per-feature NLL")

## Summary TSV

In [ ]:
rows: list[dict] = []
for ds, results in PER_DATASET_RESULTS.items():
    nll = results.get("nll_per_variant", {})
    kl = results.get("kl_per_variant", {})
    active = results.get("active_dim_per_variant", {})
    for variant, per_mod_series in nll.items():
        for modality, series in per_mod_series.items():
            subset = series
            if shared_gene_set:
                subset = series[series.index.isin(shared_gene_set)]
            row = {
                "dataset": ds,
                "variant": variant,
                "modality": modality,
                "n_features": int(len(subset)) if subset is not None else 0,
                "median_nll_markers_tfs": float(np.median(subset)) if len(subset) else float("nan"),
                "median_nll_all": float(np.median(series)),
                "mean_kl_per_dim": float(np.mean(kl.get(variant, {}).get(modality, np.nan)))
                if kl.get(variant, {}).get(modality) is not None
                else float("nan"),
                "n_active_dims": float(active.get(variant, {}).get(modality, np.nan)),
            }
            rows.append(row)

summary_df = pd.DataFrame(rows)
summary_path = Path(summary_tsv)
summary_path.parent.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(summary_path, sep="\t", index=False)
print(f"Wrote {len(summary_df)} rows to {summary_path}")
summary_df.head(20)

## D5 user step — model directory paths for wandb comparison

D5 (4-arm horseshoe comparison) is performed by the **user** on wandb (see
`Steps for user / S1` in the plan). The notebook only prints the four Plan 1
Block 3 run directories so they are easy to locate.


In [ ]:
d5_rows = [
    ("Normal(0,1) baseline", "bm_mm_zinit_vis01_softplus_dwl0_klw0"),
    ("Normal(0,1) baseline", "immune_large_ld_bs2k_zinit_vis01_softplus_dwl0_klw0"),
    ("Horseshoe LN-LN analytical", "bm_mm_zinit_vis01_softplus_horseshoe_ln_dwl0_klw0"),
    ("Horseshoe LN-LN analytical", "immune_large_ld_bs2k_zinit_vis01_softplus_horseshoe_ln_dwl0_klw0"),
    ("Horseshoe LN-LN MC", "bm_mm_zinit_vis01_softplus_horseshoe_lnmc_dwl0_klw0"),
    ("Horseshoe LN-LN MC", "immune_large_ld_bs2k_zinit_vis01_softplus_horseshoe_lnmc_dwl0_klw0"),
    ("Horseshoe LN-Gamma MC", "bm_mm_zinit_vis01_softplus_horseshoe_gm_dwl0_klw0"),
    ("Horseshoe LN-Gamma MC", "immune_large_ld_bs2k_zinit_vis01_softplus_horseshoe_gm_dwl0_klw0"),
]

print("D5 horseshoe 4-arm runs (open these on wandb):")
for arm, name in d5_rows:
    if not sweep_df.empty and (sweep_df["name"] == name).any():
        row = sweep_df[sweep_df["name"] == name].iloc[0]
        path = Path(row["results_folder"]) / "model"
        exists = (path / "model.pt").exists()
        status = "ready" if exists else "pending"
    else:
        path = "(not in TSV)"
        status = "missing"
    print(f"  [{status:7s}] {arm:32s} -> {name}")
    print(f"             {path}")